# Final NEDI x2 Evaluation — AWS Multi-CPU

Run this notebook from the repository root on the AWS server. It uses four worker processes to evaluate Set5, Set14, BSD100, and Urban100 at x2. Bicubic is rerun on the same AWS CPU for a fair timing comparison. Results are checkpointed after every image and can be resumed. The notebook still works in Colab as a fallback.

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False
    print('AWS/local Jupyter environment detected.')


AWS/local Jupyter environment detected.


In [4]:
import os
import subprocess
from pathlib import Path

# Prevent each worker from secretly creating more numerical-library threads.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'

if IN_COLAB:
    REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    # Jupyter commonly starts in notebooks/, rather than the repository root.
    start_dir = Path.cwd().resolve()
    REPO_ROOT = next((directory for directory in (start_dir, *start_dir.parents)
                      if (directory / 'app').is_dir()), None)
    if REPO_ROOT is None:
        raise RuntimeError(
            f'Could not find the repository above {start_dir}. ' 
            'Open this notebook from the repository or set its working directory to it.'
        )

os.chdir(REPO_ROOT)
print(f'Repository ready: {REPO_ROOT}')


Repository ready: /home/ubuntu/Code/SuperResolution-Comparative-Analysis


In [5]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.evaluation.data_validation import validate_prepared_dataset
from app.evaluation.bicubic import BicubicEvaluationConfig, evaluate_bicubic_image
from app.evaluation.experiment import write_results_csv
from app.evaluation.images import pair_image_paths
from app.evaluation.nedi import NEDIEvaluationConfig, evaluate_nedi_image
from app.evaluation.reporting import summarize_results

print('Bicubic and NEDI x2 evaluators imported successfully.')


Bicubic and NEDI x2 evaluators imported successfully.


In [6]:
from datetime import UTC, datetime
from concurrent.futures import ProcessPoolExecutor, as_completed
import csv

default_data_root = '/content/drive/MyDrive/FYP_SR_Data' if IN_COLAB else '/mnt/fyp-data/FYP_SR_Data'
DATA_ROOT = Path(os.environ.get('FYP_SR_DATA_ROOT', default_data_root))
WORKERS = int(os.environ.get('FYP_NEDI_WORKERS', '4'))
COMPUTE_INSTANCE = os.environ.get('FYP_COMPUTE_INSTANCE', 'm7i.2xlarge')
# Leave as None for a new run. To resume after an interruption, replace
# None with the timestamped folder name printed by the earlier run.
RESUME_RUN_ID = "20260830_162834_utc"
RUN_ID = RESUME_RUN_ID or datetime.now(UTC).strftime('%Y%m%d_%H%M%S_utc')
RUN_ROOT = DATA_ROOT / 'results' / 'final_nedi' / 'x2_full' / RUN_ID
METRICS_ROOT = RUN_ROOT / 'metrics'
DATASETS = ('Set5', 'Set14', 'BSD100', 'Urban100')

if WORKERS < 1:
    raise ValueError('FYP_NEDI_WORKERS must be at least 1.')
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}')

print(f'Full NEDI x2 run: {RUN_ROOT}')
print(f'Compute: {COMPUTE_INSTANCE}; workers: {WORKERS}')
print('Protocol: 3 warm-ups and 10 timed CPU runs per image.')


Full NEDI x2 run: /mnt/fyp-data/FYP_SR_Data/results/final_nedi/x2_full/20260830_162834_utc
Compute: m7i.2xlarge; workers: 4
Protocol: 3 warm-ups and 10 timed CPU runs per image.


In [7]:
validations = {}
for dataset in DATASETS:
    validation = validate_prepared_dataset(dataset, 2, DATA_ROOT)
    validations[dataset] = validation
    print(f'VALID: {dataset} x2 has {validation.image_count} complete HR/LR pairs.')

print('All x2 datasets passed validation.')


VALID: Set5 x2 has 5 complete HR/LR pairs.
VALID: Set14 x2 has 14 complete HR/LR pairs.
VALID: BSD100 x2 has 100 complete HR/LR pairs.
VALID: Urban100 x2 has 100 complete HR/LR pairs.
All x2 datasets passed validation.


In [8]:
def load_checkpoint(path):
    if not path.exists():
        return []
    with path.open(newline='', encoding='utf-8') as file:
        return list(csv.DictReader(file))

def run_dataset_method(dataset, method):
    validation = validations[dataset]
    pairs = pair_image_paths(validation.hr_directory, validation.lr_directory)
    checkpoint_csv = METRICS_ROOT / f'{dataset}_x2_{method}_aws_final.csv'
    records = load_checkpoint(checkpoint_csv)
    completed_images = {record['image'] for record in records}
    pending_pairs = [(hr, lr) for hr, lr in pairs if hr.name not in completed_images]

    if method == 'bicubic':
        config = BicubicEvaluationConfig(dataset=dataset, scale=2, warmup_runs=3, timed_runs=10)
        evaluator = evaluate_bicubic_image
    else:
        config = NEDIEvaluationConfig(
            dataset=dataset, scale=2, window_size=8, edge_threshold=8.0,
            warmup_runs=3, timed_runs=10,
        )
        evaluator = evaluate_nedi_image

    print(f'{dataset} x2 {method}: {len(completed_images)}/{len(pairs)} already complete.')
    with ProcessPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(evaluator, hr, lr, config): hr.name for hr, lr in pending_pairs}
        for future in as_completed(futures):
            record = future.result()
            record['compute_instance'] = COMPUTE_INSTANCE
            record['execution_mode'] = 'parallel_process_workers'
            record['worker_count'] = WORKERS
            records.append(record)
            records.sort(key=lambda item: item['image'])
            write_results_csv(records, checkpoint_csv, overwrite=True)
            print(
                f'{dataset} x2 {method}: {len(records)}/{len(pairs)} — {record["image"]} — '
                f'PSNR-Y={float(record["psnr_y"]):.4f}, '
                f'time={float(record["latency_mean_ms"]) / 1000:.2f}s'
            )
    return records

bicubic_records = []
nedi_records = []
for dataset in DATASETS:
    bicubic_records.extend(run_dataset_method(dataset, 'bicubic'))
    nedi_records.extend(run_dataset_method(dataset, 'nedi'))

print(f'Completed {len(nedi_records)} NEDI and {len(bicubic_records)} bicubic x2 evaluations.')


Set5 x2 bicubic: 5/5 already complete.
Set5 x2 nedi: 5/5 already complete.
Set14 x2 bicubic: 14/14 already complete.
Set14 x2 nedi: 14/14 already complete.
BSD100 x2 bicubic: 100/100 already complete.
BSD100 x2 nedi: 100/100 already complete.
Urban100 x2 bicubic: 100/100 already complete.
Urban100 x2 nedi: 90/100 already complete.
Urban100 x2 nedi: 91/100 — img_092.png — PSNR-Y=19.5339, time=38.76s
Urban100 x2 nedi: 92/100 — img_093.png — PSNR-Y=30.5416, time=39.42s
Urban100 x2 nedi: 93/100 — img_091.png — PSNR-Y=25.4078, time=39.95s
Urban100 x2 nedi: 94/100 — img_094.png — PSNR-Y=28.1802, time=40.36s
Urban100 x2 nedi: 95/100 — img_097.png — PSNR-Y=26.0712, time=35.17s
Urban100 x2 nedi: 96/100 — img_096.png — PSNR-Y=25.4305, time=37.50s
Urban100 x2 nedi: 97/100 — img_098.png — PSNR-Y=21.9583, time=38.15s
Urban100 x2 nedi: 98/100 — img_095.png — PSNR-Y=21.1643, time=44.03s
Urban100 x2 nedi: 99/100 — img_099.png — PSNR-Y=25.5475, time=43.51s
Urban100 x2 nedi: 100/100 — img_100.png — PSNR

In [9]:
nedi_combined_csv = METRICS_ROOT / 'nedi_x2_all_images_final.csv'
bicubic_combined_csv = METRICS_ROOT / 'bicubic_x2_aws_all_images_final.csv'
summary_csv = METRICS_ROOT / 'nedi_x2_summary_final.csv'
comparison_summary_csv = METRICS_ROOT / 'x2_aws_comparison_summary.csv'
summary_records = summarize_results(nedi_records)
comparison_summary = summarize_results(bicubic_records + nedi_records)
write_results_csv(nedi_records, nedi_combined_csv, overwrite=True)
write_results_csv(bicubic_records, bicubic_combined_csv, overwrite=True)
write_results_csv(summary_records, summary_csv, overwrite=True)
write_results_csv(comparison_summary, comparison_summary_csv, overwrite=True)

for row in summary_records:
    print(
        f"{row['dataset']} x2: images={row['image_count']}, "
        f"PSNR-Y={row['psnr_y']:.4f}, SSIM-Y={row['ssim_y']:.4f}, "
        f"PSNR-RGB={row['psnr_rgb']:.4f}, SSIM-RGB={row['ssim_rgb']:.4f}, "
        f"latency={row['latency_mean_ms']:.2f} ms"
    )

print(f'NEDI combined results: {nedi_combined_csv}')
print(f'NEDI summary: {summary_csv}')
print(f'AWS bicubic/NEDI comparison: {comparison_summary_csv}')


Set5 x2: images=5, PSNR-Y=33.1564, SSIM-Y=0.9244, PSNR-RGB=31.1912, SSIM-RGB=0.8974, latency=5749.57 ms
Set14 x2: images=14, PSNR-Y=29.8755, SSIM-Y=0.8550, PSNR-RGB=28.0457, SSIM-RGB=0.8254, latency=12856.78 ms
BSD100 x2: images=100, PSNR-Y=29.0456, SSIM-Y=0.8204, PSNR-RGB=27.6528, SSIM-RGB=0.8027, latency=8261.01 ms
Urban100 x2: images=100, PSNR-Y=26.9185, SSIM-Y=0.8370, PSNR-RGB=25.4450, SSIM-RGB=0.8209, latency=42602.28 ms
NEDI combined results: /mnt/fyp-data/FYP_SR_Data/results/final_nedi/x2_full/20260830_162834_utc/metrics/nedi_x2_all_images_final.csv
NEDI summary: /mnt/fyp-data/FYP_SR_Data/results/final_nedi/x2_full/20260830_162834_utc/metrics/nedi_x2_summary_final.csv
AWS bicubic/NEDI comparison: /mnt/fyp-data/FYP_SR_Data/results/final_nedi/x2_full/20260830_162834_utc/metrics/x2_aws_comparison_summary.csv
